# Actinver — US Equity Dividend Strategy
**Centro de control mensual** · Correr celda por celda cada mes de rebalanceo.

In [ ]:
# ============================================================
# CELDA 1 — [PARAMS]  Única celda a modificar en uso normal
# ============================================================

# Modo de ejecución
RUN_MODE = "backtest"   # "backtest" | "live"

# Período
BACKTEST_START = "2016-01-01"
BACKTEST_END   = None          # None = hasta hoy

# --- Restricciones del portafolio (QP) ---
MIN_PORTFOLIO_YIELD = 0.03
MAX_WEIGHT          = 0.05
RISK_AVERSION       = 1.0      # γ en la función objetivo

# --- Filtros del universo ---
MIN_MKTCAP       = 100e9
MIN_YIELD_SCREEN = 0.005
MIN_ADV          = 50e6

# --- Protocolo de infeasibility ---
INFEASIBILITY_MODE  = False
RELAXED_YIELD       = 0.03
RELAXED_MAX_WEIGHT  = 0.10
RELAXED_MIN_ADV     = 25e6
RELAXED_MIN_MKTCAP  = 100e9

# --- Black-Litterman ---
TAU             = 0.05
LOOKBACK_MONTHS = 36
MIN_OBS_MONTHS  = 24

# --- Stop-loss / Take-profit ---
EWMA_SPAN    = 12
SL_THRESHOLD = 1.0
TP_THRESHOLD = 1.0

print(f"RUN_MODE: {RUN_MODE}  |  {BACKTEST_START} → {BACKTEST_END or 'hoy'}")
print(f"Yield mínimo: {MIN_PORTFOLIO_YIELD:.1%}  |  Peso máx: {MAX_WEIGHT:.0%}  |  γ: {RISK_AVERSION}")

RUN_MODE: backtest  |  2016-01-01 → hoy
Yield mínimo: 3.0%  |  Peso máx: 5%  |  γ: 1.0


In [2]:
# ============================================================
# CELDA 2 — Imports y setup
# ============================================================
import importlib.util, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

ROOT = Path().resolve()
DATA_RAW       = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
SRC            = ROOT / "src"

def _load_mod(alias, fname):
    spec = importlib.util.spec_from_file_location(alias, SRC / fname)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_u = _load_mod("universe",  "01_universe.py")
_f = _load_mod("features",  "02_features.py")
_o = _load_mod("optimizer", "03_optimizer.py")
_s = _load_mod("signals",   "04_signals.py")
_b = _load_mod("backtest",  "05_backtest.py")

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})
print("Módulos cargados  ✓")

Módulos cargados  ✓


In [3]:
# ============================================================
# CELDA 3 — Universo elegible
# ============================================================

if RUN_MODE == "live":
    universe_df = _u.get_universe(
        min_mktcap=MIN_MKTCAP, min_yield=MIN_YIELD_SCREEN,
        min_adv=MIN_ADV, save=True,
    )
else:
    csvs = sorted(DATA_RAW.glob("universe_*.csv"))
    if not csvs:
        print("No hay universo guardado — corriendo screening...")
        universe_df = _u.get_universe(min_mktcap=MIN_MKTCAP, min_yield=MIN_YIELD_SCREEN, min_adv=MIN_ADV)
    else:
        universe_df = pd.read_csv(csvs[-1])
        print(f"Universo cargado: {csvs[-1].name}")

print(f"\nAcciones elegibles: {len(universe_df)}")
print(f"Market cap mediana: ${universe_df['market_cap'].median()/1e9:.0f}B")
print(f"Yield TTM promedio: {universe_df['dividend_yield'].mean():.2%}")
print(f"ADV 90d promedio:   ${universe_df['adv_90d'].mean()/1e6:.0f}M")

display(
    universe_df.style
    .format({'market_cap': '${:,.0f}', 'dividend_yield': '{:.2%}',
             'adv_90d': '${:,.0f}', 'price': '${:.2f}'})
    .background_gradient(subset=['dividend_yield'], cmap='YlGn')
    .background_gradient(subset=['market_cap'], cmap='Blues')
    .set_caption(f"Universo elegible — {pd.Timestamp.today().strftime('%Y-%m-%d')}")
)

Universo cargado: universe_20260531.csv

Acciones elegibles: 76
Market cap mediana: $182B
Yield TTM promedio: 2.14%
ADV 90d promedio:   $1688M


AttributeError: The '.style' accessor requires jinja2

In [ ]:
# ============================================================
# CELDA 4 — Features: Black-Litterman + Ledoit-Wolf
# ============================================================

if RUN_MODE == "live":
    rebalance_dt = pd.Timestamp.today().normalize()
    features_df, cov_df = _f.get_features(
        rebalance_date=rebalance_dt, universe_df=universe_df,
        lookback_months=LOOKBACK_MONTHS, min_obs_months=MIN_OBS_MONTHS,
        lambda_bl=RISK_AVERSION, tau=TAU, save=True,
    )
else:
    feat_files = sorted(DATA_PROCESSED.glob("features_*.parquet"))
    cov_files  = sorted(DATA_PROCESSED.glob("cov_*.parquet"))
    if not feat_files:
        raise FileNotFoundError("No hay features guardadas. Corre 02_features.py primero.")
    features_df = pd.read_parquet(feat_files[-1])
    cov_df      = pd.read_parquet(cov_files[-1])
    stem = cov_files[-1].stem.split("_")
    rebalance_dt = pd.Timestamp(f"{stem[1]}-{stem[2]}-01") + pd.offsets.MonthEnd(0)
    print(f"Features: {feat_files[-1].name}")

# ---- Plots ----
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle(f"Black-Litterman + Ledoit-Wolf — {rebalance_dt.strftime('%Y-%m')}", fontsize=13, y=1.01)

# 1. Correlación
S = cov_df.values
sd = np.sqrt(np.diag(S))
corr_mat = S / np.outer(sd, sd)
corr_df  = pd.DataFrame(corr_mat, index=cov_df.index, columns=cov_df.columns)
mask = np.zeros_like(corr_mat, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(corr_df, ax=axes[0], cmap='RdYlGn', vmin=-0.2, vmax=1,
            xticklabels=False, yticklabels=False, cbar_kws={'shrink': 0.8})
axes[0].set_title(f"Correlación Ledoit-Wolf\n({len(cov_df)} stocks)")

# 2. μ_BL bar chart
fs = features_df.sort_values('mu_bl')
cols = ['#d73027' if v < 0 else '#1a9850' for v in fs['mu_bl']]
axes[1].barh(fs['ticker'], fs['mu_bl'] * 12, color=cols, edgecolor='none')
axes[1].axvline(0, color='black', lw=0.8)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
axes[1].set_xlabel('μ_BL anualizado')
axes[1].set_title('Retornos esperados B-L')
axes[1].tick_params(labelsize=7)

# 3. Yield TTM vs μ_BL
axes[2].scatter(features_df['dividend_yield_ttm']*100, features_df['mu_bl']*12*100,
                alpha=0.7, edgecolors='navy', linewidths=0.5, s=55)
for _, r in features_df.iterrows():
    axes[2].annotate(r['ticker'],
                     (r['dividend_yield_ttm']*100, r['mu_bl']*12*100),
                     fontsize=6, alpha=0.75)
axes[2].axhline(0, color='gray', lw=0.5)
axes[2].set_xlabel('Dividend Yield TTM (%)')
axes[2].set_ylabel('μ_BL anualizado (%)')
axes[2].set_title('View (yield) → Posterior (μ_BL)')

plt.tight_layout()
plt.show()

# Tabla comparativa
comp = features_df[['ticker','dividend_yield_ttm','mu_bl']].copy()
comp['mu_bl_anual'] = comp['mu_bl'] * 12
comp = comp.drop(columns='mu_bl').sort_values('mu_bl_anual', ascending=False)
print(f"\nEstadísticas μ_BL (anualizado):  media={comp['mu_bl_anual'].mean():.2%}  min={comp['mu_bl_anual'].min():.2%}  max={comp['mu_bl_anual'].max():.2%}")
display(
    comp.style.format({'dividend_yield_ttm': '{:.2%}', 'mu_bl_anual': '{:.2%}'})
    .background_gradient(subset=['dividend_yield_ttm'], cmap='YlGn')
    .background_gradient(subset=['mu_bl_anual'], cmap='RdYlGn')
    .set_caption("View vs Posterior B-L")
)

In [ ]:
# ============================================================
# CELDA 5 — Optimización QP
# ============================================================

def _run_optimizer(feat, cov, yield_c, max_w, gamma_c, label=""):
    return _o.optimize_portfolio(
        features_df=feat, cov_df=cov,
        gamma=gamma_c, min_yield=yield_c, max_weight=max_w,
        rebalance_date=rebalance_dt, save=False,
    )

weights_df = _run_optimizer(features_df, cov_df, MIN_PORTFOLIO_YIELD, MAX_WEIGHT, RISK_AVERSION)

# ---- Modo infeasibility ----
if weights_df is None or INFEASIBILITY_MODE:
    print("\n" + "─"*60)
    print("DIAGNÓSTICO DE INFEASIBILITY")
    print("─"*60)
    dy = features_df.set_index('ticker')['dividend_yield_ttm']
    max_y, _ = _o._max_achievable_yield(dy.values, MAX_WEIGHT)
    print(f"  Yield máximo alcanzable (sin constraint):  {max_y:.2%}")
    print(f"  N° stocks elegibles:                       {len(features_df)}")
    print()
    scenarios = {
        "A — Relajar yield objetivo":    _run_optimizer(features_df, cov_df, RELAXED_YIELD, MAX_WEIGHT, RISK_AVERSION),
        "B — Relajar peso máximo":        _run_optimizer(features_df, cov_df, MIN_PORTFOLIO_YIELD, RELAXED_MAX_WEIGHT, RISK_AVERSION),
        "C — Sin constraint yield":       _run_optimizer(features_df, cov_df, 0.0, MAX_WEIGHT, RISK_AVERSION),
        "D — Mantener portafolio anterior": None,
    }
    for name, sol in scenarios.items():
        if sol is not None:
            y = sol['contribution_yield'].sum()
            n = (sol['weight'] > 1e-4).sum()
            print(f"  {name:<35} → factible  yield={y:.2%}  n={n}")
        else:
            print(f"  {name:<35} → mantener posición")
    print("─"*60)
    print("Selecciona el escenario ajustando RELAXED_* en PARAMS y re-corriendo esta celda.")
    if weights_df is None:
        raise RuntimeError("QP infeasible con parámetros actuales — ajusta INFEASIBILITY_MODE params.")

# ---- Limpiar pesos residuales ----
weights_df = weights_df[weights_df['weight'] > 1e-4].copy()
port_yield = weights_df['contribution_yield'].sum()
S_active   = cov_df.loc[weights_df['ticker'], weights_df['ticker']].values
port_vol   = np.sqrt(weights_df['weight'].values @ S_active @ weights_df['weight'].values * 12)

print(f"Portafolio óptimo — {rebalance_dt.strftime('%Y-%m')}")
print(f"  Yield esperado:  {port_yield:.2%}  (mínimo: {MIN_PORTFOLIO_YIELD:.1%})")
print(f"  Volatilidad:     {port_vol:.2%}  anualizada")
print(f"  Acciones:        {len(weights_df)}")

# ---- Plots ----
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle(f"Portafolio óptimo — {rebalance_dt.strftime('%Y-%m')}", fontsize=13, y=1.01)

# 1. Pesos
w_plot = weights_df.sort_values('weight')
axes[0].barh(w_plot['ticker'], w_plot['weight']*100, color='steelblue', edgecolor='none')
axes[0].axvline(MAX_WEIGHT*100, color='red', lw=1, ls='--', label=f'Máx {MAX_WEIGHT:.0%}')
axes[0].set_xlabel('Peso (%)')
axes[0].set_title('Pesos óptimos')
axes[0].legend(fontsize=8)

# 2. Contribución al yield
w_plot2 = weights_df.sort_values('contribution_yield')
axes[1].barh(w_plot2['ticker'], w_plot2['contribution_yield']*100, color='#2ca02c', edgecolor='none')
axes[1].axvline(0, color='black', lw=0.5)
axes[1].axvline(MIN_PORTFOLIO_YIELD/len(weights_df)*100, color='red', lw=0.8, ls=':', label='Media objetivo')
axes[1].set_xlabel('Contribución yield (%)')
axes[1].set_title(f'Yield total: {port_yield:.2%}')
axes[1].legend(fontsize=8)

# 3. Peso vs yield por acción
axes[2].scatter(weights_df['contribution_yield']*100/weights_df['weight'],
                weights_df['weight']*100, s=80, alpha=0.8, edgecolors='navy', lw=0.5)
for _, r in weights_df.iterrows():
    axes[2].annotate(r['ticker'],
                     (r['contribution_yield']*100/r['weight'], r['weight']*100),
                     fontsize=7, alpha=0.8)
axes[2].axvline(MIN_PORTFOLIO_YIELD*100, color='red', lw=0.8, ls='--', label=f'Yield mínimo {MIN_PORTFOLIO_YIELD:.0%}')
axes[2].set_xlabel('Dividend Yield TTM (%)')
axes[2].set_ylabel('Peso (%)')
axes[2].set_title('Peso vs Yield por acción')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

display(
    weights_df[['ticker','weight','expected_return','contribution_yield']].style
    .format({'weight': '{:.2%}', 'expected_return': '{:.3%}', 'contribution_yield': '{:.4%}'})
    .background_gradient(subset=['weight'], cmap='Blues')
    .background_gradient(subset=['contribution_yield'], cmap='YlGn')
    .set_caption(f"Pesos óptimos — yield portafolio: {port_yield:.2%}")
)

In [ ]:
# ============================================================
# CELDA 6 — Señales stop-loss / take-profit
# ============================================================

sig_files = sorted(DATA_PROCESSED.glob("signals_*.parquet"))
if sig_files:
    signals_df = pd.read_parquet(sig_files[-1])
    print(f"Señales cargadas: {sig_files[-1].name}")
else:
    # Construir posiciones desde weights + precios de entrada
    import yfinance as yf
    tks = weights_df['ticker'].tolist()
    raw = yf.download(tks, period='2d', auto_adjust=True, progress=False)
    prices_entry = raw['Close'].iloc[-1] if isinstance(raw['Close'], pd.DataFrame) else raw['Close']
    positions_records = []
    for _, row in weights_df.iterrows():
        tk = row['ticker']
        ep = float(prices_entry.get(tk, 0)) if hasattr(prices_entry, 'get') else 0.0
        positions_records.append({'ticker': tk, 'entry_date': rebalance_dt, 'entry_price': ep})
    positions_df = pd.DataFrame(positions_records)
    signals_df = _s.evaluate_signals(
        positions_df=positions_df, eval_date=rebalance_dt,
        sl_threshold=SL_THRESHOLD, tp_threshold=TP_THRESHOLD,
        ewma_span=EWMA_SPAN, save=True,
    )

# ---- Display ----
n_hold = (signals_df['signal'] == 'hold').sum()
n_stop = (signals_df['signal'] == 'stop').sum()
n_take = (signals_df['signal'] == 'take').sum()

print(f"\nResumen señales — {rebalance_dt.strftime('%Y-%m')}")
print(f"  HOLD: {n_hold}   STOP-LOSS: {n_stop}   TAKE-PROFIT: {n_take}")

def _color_signal(val):
    colors = {'hold': 'background-color: #f0f0f0', 'stop': 'background-color: #ffcccc', 'take': 'background-color: #ccffcc'}
    return colors.get(val, '')

display(
    signals_df.style
    .format({'r_acum': '{:.2%}', 'sigma_ewma': '{:.2%}'})
    .applymap(_color_signal, subset=['signal'])
    .set_caption("Señales stop-loss / take-profit")
)

if n_stop + n_take > 0:
    triggered = signals_df[signals_df['signal'] != 'hold']
    print(f"\n⚠  {n_stop + n_take} posiciones a liquidar en el siguiente rebalanceo:")
    for _, r in triggered.iterrows():
        print(f"   {r['ticker']:<6}  {r['signal'].upper():<5}  r_acum={r['r_acum']:+.2%}  σ={r['sigma_ewma']:.2%}")

In [ ]:
# ============================================================
# CELDA 7 — Backtest completo  (solo RUN_MODE == "backtest")
# ============================================================

if RUN_MODE != "backtest":
    print("Celda 7 solo ejecuta en RUN_MODE='backtest'")
else:
    res_file = DATA_PROCESSED / "backtest_results.parquet"
    if res_file.exists():
        results = pd.read_parquet(res_file)
        print(f"Resultados cargados: {res_file.name}  ({len(results)} meses)")
    else:
        print("Corriendo backtest completo (puede tardar ~15 min)...")
        results, _ = _b.run_backtest(
            backtest_start=BACKTEST_START,
            min_portfolio_yield=MIN_PORTFOLIO_YIELD,
            max_weight=MAX_WEIGHT,
            gamma=RISK_AVERSION,
        )

    fig, axes = plt.subplots(3, 1, figsize=(15, 13), sharex=True)
    fig.suptitle("Backtest — Actinver US Equity Dividend Strategy", fontsize=14, y=1.01)

    # 1. NAV portafolio vs benchmark
    ax = axes[0]
    ax.plot(results.index, results['nav'], label='Portafolio', color='#1f77b4', lw=2)
    # Reconstruir NAV benchmark
    bm_nav = 100 * (1 + results['benchmark_return']).cumprod()
    ax.plot(results.index, bm_nav, label='SPY (benchmark)', color='#ff7f0e', lw=1.5, ls='--')
    ax.set_ylabel('NAV (base 100)')
    ax.set_title('NAV portafolio vs SPY')
    ax.legend()
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:,.0f}'))

    # 2. Dividend yield realizado mensual
    ax = axes[1]
    monthly_yield_ann = results['realized_yield'] * 12
    ax.bar(results.index, monthly_yield_ann * 100, width=20, color='#2ca02c', alpha=0.7, label='Yield realizado (anualizado)')
    ax.axhline(MIN_PORTFOLIO_YIELD * 100, color='red', lw=1.5, ls='--', label=f'Objetivo {MIN_PORTFOLIO_YIELD:.0%}')
    ax.set_ylabel('Dividend yield (%)')
    ax.set_title('Dividend yield realizado mensual (anualizado)')
    ax.legend()

    # 3. Drawdown
    ax = axes[2]
    rolling_max = results['nav'].cummax()
    drawdown = (results['nav'] / rolling_max - 1) * 100
    ax.fill_between(results.index, drawdown, 0, alpha=0.4, color='red')
    ax.plot(results.index, drawdown, color='red', lw=0.8)
    ax.set_ylabel('Drawdown (%)')
    ax.set_title(f"Drawdown  (máx: {drawdown.min():.1f}%)")
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.0f}%'))

    plt.tight_layout()
    plt.show()

    # Heatmap de número de stocks activos por mes
    fig2, ax2 = plt.subplots(figsize=(15, 2.5))
    n_stocks = results['n_stocks'].values.reshape(1, -1)
    im = ax2.imshow(n_stocks, aspect='auto', cmap='Blues', vmin=0, vmax=25)
    ax2.set_yticks([])
    ax2.set_xticks(range(0, len(results), 12))
    ax2.set_xticklabels([results.index[i].strftime('%Y') for i in range(0, len(results), 12)], rotation=0)
    ax2.set_title('Número de stocks activos por mes')
    plt.colorbar(im, ax=ax2, shrink=0.8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# CELDA 8 — Reporte de métricas
# ============================================================
import json

met_file = DATA_PROCESSED / "backtest_metrics.json"
if met_file.exists():
    with open(met_file) as f:
        metrics = json.load(f)
else:
    if RUN_MODE == "backtest" and 'results' in dir():
        metrics = _b.compute_metrics(results)
    else:
        raise FileNotFoundError("Corre el backtest primero (celda 7).")

# Formatear tabla
rows = [
    ("MÉTRICAS DEL MANDATO",         "",          ""),
    ("CAGR portafolio",               f"{metrics['CAGR_portfolio']:.2%}",    f"{metrics['CAGR_benchmark']:.2%}"),
    ("Volatilidad anualizada",        f"{metrics['Vol_annual_portfolio']:.2%}", "—"),
    ("Sharpe Ratio",                  f"{metrics['Sharpe_Ratio']:.3f}",       "—"),
    ("Sortino Ratio",                 str(round(metrics['Sortino_Ratio'],3)) if metrics['Sortino_Ratio'] else '—', "—"),
    ("Calmar Ratio",                  str(round(metrics['Calmar_Ratio'],3))  if metrics['Calmar_Ratio']  else '—', "—"),
    ("Max Drawdown",                  f"{metrics['Max_Drawdown']:.2%}",       "—"),
    ("VaR 95% mensual",               f"{metrics['VaR_95_monthly']:.2%}",     "—"),
    ("VaR 99% mensual",               f"{metrics['VaR_99_monthly']:.2%}",     "—"),
    ("CVaR 95% mensual",              f"{metrics['CVaR_95_monthly']:.2%}",    "—"),
    ("CVaR 99% mensual",              f"{metrics['CVaR_99_monthly']:.2%}",    "—"),
    ("Upper Partial Moment",          f"{metrics['Upper_Partial_Moment']:.5f}", "—"),
    ("Pain Index",                    f"{metrics['Pain_Index']:.4f}",          "—"),
    ("Pain-Gain Ratio",               str(round(metrics['Pain_Gain_Ratio'],3)) if metrics['Pain_Gain_Ratio'] else '—', "—"),
    ("",                              "",          ""),
    ("MÉTRICAS COMPLEMENTARIAS",      "",          ""),
    ("Alpha anual vs SPY",            f"{metrics['Alpha_annual']:.2%}",        "—"),
    ("Tracking Error",                f"{metrics['Tracking_Error']:.2%}",      "—"),
    ("Information Ratio",             str(round(metrics['Information_Ratio'],3)) if metrics['Information_Ratio'] else '—', "—"),
    ("Beta vs SPY",                   str(round(metrics['Beta'],3)) if metrics['Beta'] else '—', "—"),
    ("",                              "",          ""),
    ("YIELD",                         "",          ""),
    ("Yield realizado anual (promedio)", f"{metrics['Avg_Realized_Yield_annual']:.2%}", f"Objetivo: {MIN_PORTFOLIO_YIELD:.0%}"),
    ("% meses con yield ≥ 3%",        f"{metrics['Pct_months_yield_met']:.1%}",   "—"),
    ("",                              "",          ""),
    ("OPERACIONES",                   "",          ""),
    ("Turnover mensual promedio",      f"{metrics['Avg_Monthly_Turnover']:.2%}" if metrics['Avg_Monthly_Turnover'] else '—', "—"),
    ("Activaciones stop-loss",        str(metrics['N_StopLoss_activations']),    "—"),
    ("Activaciones take-profit",      str(metrics['N_TakeProfit_activations']),  "—"),
    ("N meses backtest",              str(metrics['N_months']),                  "—"),
]

df_met = pd.DataFrame(rows, columns=['Métrica', 'Portafolio', 'Benchmark / Referencia'])

def _style_row(row):
    if row['Benchmark / Referencia'] == '' and row['Portafolio'] == '':
        return ['font-weight: bold; background-color: #e8e8e8'] * 3
    return [''] * 3

display(
    df_met.style
    .apply(_style_row, axis=1)
    .hide(axis='index')
    .set_caption(f"Métricas de desempeño — backtest {BACKTEST_START} → {results.index[-1].strftime('%Y-%m') if 'results' in dir() else ''}")
)

In [ ]:
# ============================================================
# CELDA 9 — Validación cruzada walk-forward
# ============================================================
# Calcula calidad de predicción de μ_BL vs retornos realizados.
# Para cada mes t: μ_BL(t-1) como predicción del retorno de t.

wf_file = DATA_PROCESSED / "walk_forward_results.parquet"

if wf_file.exists():
    wf = pd.read_parquet(wf_file)
    print(f"Walk-forward cargado: {wf_file.name}  ({len(wf)} meses)")
else:
    print("Computando walk-forward desde precios cacheados...")
    if 'results' not in dir():
        raise RuntimeError("Corre el backtest primero (celda 7).")

    import yfinance as yf
    # Descargar precios mensuales una vez
    tks_all = universe_df['ticker'].tolist()
    raw_wf = yf.download(tks_all, start="2013-01-01", auto_adjust=True, progress=True)
    cl_wf  = raw_wf['Close'] if isinstance(raw_wf['Close'], pd.DataFrame) else raw_wf[['Close']]
    mo_cl  = cl_wf.resample('ME').last()
    mo_ret = np.log(mo_cl / mo_cl.shift(1))

    wf_records = []
    months = results.index.tolist()
    for i in range(1, len(months)):
        prev_dt = months[i-1]
        curr_dt = months[i]
        wend    = prev_dt
        wstart  = wend - pd.DateOffset(months=LOOKBACK_MONTHS+1)
        ret_w   = mo_ret.loc[wstart:wend]

        available = [t for t in tks_all if t in ret_w.columns and ret_w[t].notna().sum() >= MIN_OBS_MONTHS]
        if len(available) < 5:
            continue
        returns_clean = ret_w[available].dropna()
        if returns_clean.shape[0] < MIN_OBS_MONTHS:
            continue

        try:
            sigma = _f._ledoit_wolf(returns_clean)
            shrinkage = None  # LedoitWolf().shrinkage_ not easily accessible without re-fitting
            from sklearn.covariance import LedoitWolf
            lw_tmp = LedoitWolf().fit(returns_clean.values)
            shrinkage = lw_tmp.shrinkage_

            univ_sub = universe_df[universe_df['ticker'].isin(available)].set_index('ticker')
            mc  = np.array([univ_sub.loc[t, 'market_cap'] for t in available if t in univ_sub.index], dtype=float)
            dy  = np.array([univ_sub.loc[t, 'dividend_yield'] for t in available if t in univ_sub.index], dtype=float)
            available2 = [t for t in available if t in univ_sub.index]
            sigma2 = sigma[:len(available2), :len(available2)]
            w_mkt = mc / mc.sum()
            mu_bl = _f._black_litterman(sigma2, w_mkt, dy, RISK_AVERSION, TAU)

            # Retorno realizado en curr_dt
            actual = mo_ret.loc[curr_dt, available2] if curr_dt in mo_ret.index else pd.Series(dtype=float)
            if actual.empty:
                continue

            pred   = pd.Series(mu_bl, index=available2)
            common = pred.index.intersection(actual.dropna().index)
            if len(common) < 3:
                continue
            mae  = float(np.abs(pred[common] - actual[common]).mean())
            rmse = float(np.sqrt(((pred[common] - actual[common])**2).mean()))

            # Hit ratio top tercil
            n3    = max(1, len(common)//3)
            top_pred   = set(pred[common].nlargest(n3).index)
            top_actual = set(actual[common].nlargest(n3).index)
            hit = len(top_pred & top_actual) / n3

            wf_records.append({
                'date': curr_dt, 'mae': mae, 'rmse': rmse,
                'hit_ratio': hit, 'shrinkage': shrinkage, 'n_stocks': len(common)
            })
        except Exception:
            continue

    wf = pd.DataFrame(wf_records).set_index('date')
    wf.to_parquet(wf_file)
    print(f"Walk-forward completado: {len(wf)} meses")

# ---- Display ----
print(f"\nResumen walk-forward ({len(wf)} meses):")
print(f"  MAE predicción μ_BL:       {wf['mae'].mean():.4f}  (mensual)")
print(f"  RMSE predicción μ_BL:      {wf['rmse'].mean():.4f}  (mensual)")
print(f"  Hit ratio top tercil:      {wf['hit_ratio'].mean():.1%}  (aleatorio = 33%)")
if 'shrinkage' in wf.columns:
    print(f"  Shrinkage Ledoit-Wolf avg: {wf['shrinkage'].mean():.3f}  (rango: {wf['shrinkage'].min():.3f}–{wf['shrinkage'].max():.3f})")

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle("Validación walk-forward — Calidad predictiva μ_BL", fontsize=13)

axes[0,0].plot(wf.index, wf['mae'], color='steelblue', lw=1.2)
axes[0,0].fill_between(wf.index, wf['mae'], alpha=0.2, color='steelblue')
axes[0,0].set_title('MAE predicción mensual')
axes[0,0].set_ylabel('MAE')

axes[0,1].plot(wf.index, wf['rmse'], color='coral', lw=1.2)
axes[0,1].fill_between(wf.index, wf['rmse'], alpha=0.2, color='coral')
axes[0,1].set_title('RMSE predicción mensual')

axes[1,0].plot(wf.index, wf['hit_ratio']*100, color='green', lw=1.2)
axes[1,0].axhline(33.3, color='red', lw=1, ls='--', label='Aleatorio (33%)')
axes[1,0].set_title('Hit ratio top tercil (%)')
axes[1,0].set_ylabel('%')
axes[1,0].legend()

if 'shrinkage' in wf.columns:
    axes[1,1].plot(wf.index, wf['shrinkage'], color='purple', lw=1.2)
    axes[1,1].fill_between(wf.index, wf['shrinkage'], alpha=0.15, color='purple')
    axes[1,1].set_title('Shrinkage coefficient Ledoit-Wolf')
    axes[1,1].set_ylabel('Coeficiente')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELDA 10 — Exportar reporte Excel
# ============================================================
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

timestamp = pd.Timestamp.today().strftime('%Y%m%d')
out_path  = ROOT / "reports" / f"reporte_{timestamp}.xlsx"
(ROOT / "reports").mkdir(exist_ok=True)

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:

    # Hoja 1: Métricas
    met_df = pd.DataFrame(list(metrics.items()), columns=['Métrica','Valor'])
    met_df.to_excel(writer, sheet_name='Metricas', index=False)

    # Hoja 2: Pesos actuales
    weights_df[['ticker','weight','contribution_yield']].to_excel(
        writer, sheet_name='Pesos', index=False)

    # Hoja 3: NAV
    if 'results' in dir():
        results[['nav','portfolio_return','benchmark_return']].to_excel(
            writer, sheet_name='NAV')

    # Hoja 4: Dividends
    if 'results' in dir():
        results[['realized_yield']].assign(
            realized_yield_annual=results['realized_yield']*12
        ).to_excel(writer, sheet_name='Dividends')

    # Hoja 5: Infeasibility log (si existe)
    inf_log = DATA_PROCESSED / 'infeasibility_log.csv'
    if inf_log.exists():
        pd.read_csv(inf_log).to_excel(writer, sheet_name='Infeasibility', index=False)

    # Hoja 6: Walk-forward
    if 'wf' in dir():
        wf.to_excel(writer, sheet_name='WalkForward')

print(f"Reporte guardado → {out_path}")
print(f"  Hojas: Metricas, Pesos, NAV, Dividends" + (', Infeasibility' if inf_log.exists() else '') + (', WalkForward' if 'wf' in dir() else ''))